# Notebook 02 — Data Cleaning

## Objectif

Ce notebook a pour objectif de nettoyer les données brutes du projet de rééducation motrice.

Les opérations réalisées sont :

- suppression des doublons réellement identifiés ;
- traitement des valeurs manquantes ;
- correction des types de données ;
- correction des valeurs invalides ;
- vérification de la cohérence entre les tables ;
- vérification des dates et de la chronologie ;
- vérification de la variable `progression` ;
- sauvegarde des données nettoyées dans le dossier `data/processed/`.

Les données originales présentes dans `data/raw/` ne sont jamais modifiées.

Le résultat de ce notebook sera utilisé dans les notebooks suivants :
- Data Integration
- EDA
- KPI Analysis
- Progression Analysis
- Dashboard

In [35]:
import pandas as pd
import numpy as np
import os

In [36]:
RAW_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

In [37]:
print(os.getcwd())
print(os.path.abspath(PROCESSED_PATH))
print(os.path.exists(PROCESSED_PATH))
print(os.path.isdir(PROCESSED_PATH))

c:\Users\eyato\OneDrive\Desktop\Data_projects\KineKids-AI\data_analysis\notebooks
c:\Users\eyato\OneDrive\Desktop\Data_projects\KineKids-AI\data_analysis\data\processed
True
True


In [38]:
os.makedirs(PROCESSED_PATH, exist_ok=True)

In [39]:
patients = pd.read_csv(RAW_PATH + "patients.csv")
games = pd.read_csv(RAW_PATH + "games.csv")
exercises = pd.read_csv(RAW_PATH + "exercises.csv")
exercise_levels = pd.read_csv(RAW_PATH + "exercise_levels.csv")
patient_exercises = pd.read_csv(RAW_PATH + "patient_exercises.csv")
level_changes = pd.read_csv(RAW_PATH + "level_changes.csv")
sessions = pd.read_csv(RAW_PATH + "sessions.csv")
results = pd.read_csv(RAW_PATH + "results.csv")

In [40]:
datasets = {
    "patients": patients,
    "games": games,
    "exercises": exercises,
    "exercise_levels": exercise_levels,
    "patient_exercises": patient_exercises,
    "level_changes": level_changes,
    "sessions": sessions,
    "results": results
}

In [41]:
for name, df in datasets.items():
    print(
        f"{name:20} → "
        f"{df.shape[0]:4} lignes | "
        f"{df.shape[1]:2} colonnes"
    )

patients             →   20 lignes |  7 colonnes
games                →    3 lignes |  4 colonnes
exercises            →    8 lignes |  4 colonnes
exercise_levels      →   24 lignes |  7 colonnes
patient_exercises    →   43 lignes |  8 colonnes
level_changes        →   73 lignes |  8 colonnes
sessions             →  500 lignes |  6 colonnes
results              →  500 lignes |  5 colonnes


In [42]:
cleaned_datasets = {
    name: df.copy()
    for name, df in datasets.items()
}

In [43]:
cleaned_datasets["patients"]["date_of_birth"] = pd.to_datetime(
    cleaned_datasets["patients"]["date_of_birth"],
    errors="coerce"
)

cleaned_datasets["patients"]["created_at"] = pd.to_datetime(
    cleaned_datasets["patients"]["created_at"],
    errors="coerce"
)

cleaned_datasets["patient_exercises"]["assigned_at"] = pd.to_datetime(
    cleaned_datasets["patient_exercises"]["assigned_at"],
    errors="coerce"
)

cleaned_datasets["level_changes"]["changed_at"] = pd.to_datetime(
    cleaned_datasets["level_changes"]["changed_at"],
    errors="coerce"
)

cleaned_datasets["sessions"]["session_date"] = pd.to_datetime(
    cleaned_datasets["sessions"]["session_date"],
    errors="coerce"
)

In [44]:
date_columns = {
    "patients": ["date_of_birth", "created_at"],
    "patient_exercises": ["assigned_at"],
    "level_changes": ["changed_at"],
    "sessions": ["session_date"]
}

for table_name, columns in date_columns.items():
    print(f"\n===== {table_name.upper()} =====")
    
    for column in columns:
        print(
            f"{column}: "
            f"{cleaned_datasets[table_name][column].isna().sum()} "
            f"date(s) invalides ou manquantes"
        )


===== PATIENTS =====
date_of_birth: 0 date(s) invalides ou manquantes
created_at: 0 date(s) invalides ou manquantes

===== PATIENT_EXERCISES =====
assigned_at: 0 date(s) invalides ou manquantes

===== LEVEL_CHANGES =====
changed_at: 0 date(s) invalides ou manquantes

===== SESSIONS =====
session_date: 0 date(s) invalides ou manquantes


In [45]:
for name, df in cleaned_datasets.items():
    duplicates = df.duplicated().sum()
    
    print(f"{name:20} → {duplicates} doublon(s)")

patients             → 0 doublon(s)
games                → 0 doublon(s)
exercises            → 0 doublon(s)
exercise_levels      → 0 doublon(s)
patient_exercises    → 0 doublon(s)
level_changes        → 0 doublon(s)
sessions             → 0 doublon(s)
results              → 0 doublon(s)


In [46]:
for name in cleaned_datasets:
    cleaned_datasets[name] = cleaned_datasets[name].drop_duplicates()

In [47]:
primary_keys = {
    "patients": "id",
    "games": "id",
    "exercises": "id",
    "exercise_levels": "id",
    "patient_exercises": "id",
    "level_changes": "id",
    "sessions": "id",
    "results": "id"
}

for name, pk in primary_keys.items():
    df = cleaned_datasets[name]
    
    print(
        f"{name:20} | "
        f"Doublons PK = {df[pk].duplicated().sum()} | "
        f"NULL PK = {df[pk].isna().sum()}"
    )

patients             | Doublons PK = 0 | NULL PK = 0
games                | Doublons PK = 0 | NULL PK = 0
exercises            | Doublons PK = 0 | NULL PK = 0
exercise_levels      | Doublons PK = 0 | NULL PK = 0
patient_exercises    | Doublons PK = 0 | NULL PK = 0
level_changes        | Doublons PK = 0 | NULL PK = 0
sessions             | Doublons PK = 0 | NULL PK = 0
results              | Doublons PK = 0 | NULL PK = 0


In [48]:
for name, df in cleaned_datasets.items():
    print(f"\n===== {name.upper()} =====")
    print(df.isnull().sum())


===== PATIENTS =====
id               0
first_name       0
last_name        0
date_of_birth    0
patient_code     0
therapist_id     0
created_at       0
dtype: int64

===== GAMES =====
id             0
name           0
body_part      0
description    0
dtype: int64

===== EXERCISES =====
id             0
game_id        0
name           0
description    0
dtype: int64

===== EXERCISE_LEVELS =====
id                      0
exercise_id             0
level_number            0
title                   0
target_angle            0
required_score          0
required_progression    0
dtype: int64

===== PATIENT_EXERCISES =====
id                  0
patient_id          0
exercise_id         0
start_level_id      0
current_level_id    0
assigned_by         0
assigned_at         0
is_active           0
dtype: int64

===== LEVEL_CHANGES =====
id                      0
patient_exercise_id     0
from_level_id          43
to_level_id             0
reason                  0
session_id             45
c

In [49]:
patients_clean = cleaned_datasets["patients"]

patients_clean[
    patients_clean["first_name"].isna() |
    patients_clean["last_name"].isna() |
    patients_clean["date_of_birth"].isna() |
    patients_clean["patient_code"].isna()
]

,id,first_name,last_name,date_of_birth,patient_code,therapist_id,created_at


In [50]:
for name, df in cleaned_datasets.items():
    
    text_columns = df.select_dtypes(include=["object"]).columns
    
    for column in text_columns:
        cleaned_datasets[name][column] = (
            cleaned_datasets[name][column]
            .str.strip()
        )

In [51]:
sessions_clean = cleaned_datasets["sessions"]

invalid_scores = sessions_clean[
    (sessions_clean["score"] < 0) |
    (sessions_clean["score"] > 100)
]

print(f"Scores invalides : {len(invalid_scores)}")

Scores invalides : 0


In [52]:
invalid_duration = sessions_clean[
    sessions_clean["duration"] <= 0
]

print(f"Durées invalides : {len(invalid_duration)}")

Durées invalides : 0


In [53]:
results_clean = cleaned_datasets["results"]

invalid_success_rate = results_clean[
    (results_clean["success_rate"] < 0) |
    (results_clean["success_rate"] > 100)
]

print(
    f"Taux de réussite invalides : "
    f"{len(invalid_success_rate)}"
)

Taux de réussite invalides : 0


In [54]:
invalid_repetitions = results_clean[
    results_clean["repetitions"] < 0
]

print(
    f"Nombre de répétitions invalides : "
    f"{len(invalid_repetitions)}"
)

Nombre de répétitions invalides : 0


In [55]:
results["progression"] = results["progression"].clip(0, 100)

In [56]:
progression_outliers = results_clean[
    results_clean["progression"].notna() &
    (
        (results_clean["progression"] > 100) |
        (results_clean["progression"] < -100)
    )
]

print(
    f"Progressions très élevées/faibles : "
    f"{len(progression_outliers)}"
)

Progressions très élevées/faibles : 1


In [57]:
session_dates = cleaned_datasets["sessions"].merge(
    cleaned_datasets["patient_exercises"][
        ["id", "assigned_at"]
    ],
    left_on="patient_exercise_id",
    right_on="id",
    suffixes=("_session", "_assignment")
)

invalid_chronology = session_dates[
    session_dates["session_date"] < session_dates["assigned_at"]
]

print(
    f"Sessions avant affectation : "
    f"{len(invalid_chronology)}"
)

Sessions avant affectation : 0


In [58]:
invalid_exercise_games = cleaned_datasets["exercises"][
    ~cleaned_datasets["exercises"]["game_id"].isin(
        cleaned_datasets["games"]["id"]
    )
]

print(
    f"Exercises avec game_id invalide : "
    f"{len(invalid_exercise_games)}"
)

Exercises avec game_id invalide : 0


In [59]:
invalid_patient_exercises = cleaned_datasets["patient_exercises"][
    ~cleaned_datasets["patient_exercises"]["patient_id"].isin(
        cleaned_datasets["patients"]["id"]
    )
]

print(
    f"Patient exercises avec patient_id invalide : "
    f"{len(invalid_patient_exercises)}"
)

Patient exercises avec patient_id invalide : 0


In [60]:
invalid_sessions = cleaned_datasets["sessions"][
    ~cleaned_datasets["sessions"]["patient_exercise_id"].isin(
        cleaned_datasets["patient_exercises"]["id"]
    )
]

print(
    f"Sessions avec patient_exercise_id invalide : "
    f"{len(invalid_sessions)}"
)

Sessions avec patient_exercise_id invalide : 0


In [61]:
for name, df in cleaned_datasets.items():
    output_path = PROCESSED_PATH + f"{name}.csv"
    df.to_csv(output_path, index=False)
    
    print(f"{name}.csv → sauvegardé")

patients.csv → sauvegardé
games.csv → sauvegardé
exercises.csv → sauvegardé
exercise_levels.csv → sauvegardé
patient_exercises.csv → sauvegardé
level_changes.csv → sauvegardé
sessions.csv → sauvegardé
results.csv → sauvegardé


In [62]:
for name, df in cleaned_datasets.items():
    print(
        f"{name:20} → "
        f"{df.shape[0]:4} lignes | "
        f"{df.shape[1]:2} colonnes"
    )

patients             →   20 lignes |  7 colonnes
games                →    3 lignes |  4 colonnes
exercises            →    8 lignes |  4 colonnes
exercise_levels      →   24 lignes |  7 colonnes
patient_exercises    →   43 lignes |  8 colonnes
level_changes        →   73 lignes |  8 colonnes
sessions             →  500 lignes |  6 colonnes
results              →  500 lignes |  5 colonnes


In [ ]:
for name, df in cleaned_datasets.items():
    print(
        f"{name:20} → "
        f"{df.duplicated().sum()} doublon(s)"
    )

patients             → 0 doublon(s)
games                → 0 doublon(s)
exercises            → 0 doublon(s)
exercise_levels      → 0 doublon(s)
patient_exercises    → 0 doublon(s)
level_changes        → 0 doublon(s)
sessions             → 0 doublon(s)
results              → 0 doublon(s)


: 

# Conclusion

Le nettoyage des données a permis de :

- corriger les types de données ;
- traiter les doublons identifiés ;
- vérifier les valeurs manquantes ;
- contrôler les valeurs numériques ;
- vérifier les scores et taux de réussite ;
- contrôler les répétitions et les durées ;
- vérifier la cohérence des relations entre les tables ;
- vérifier la chronologie des sessions ;
- contrôler les valeurs de progression ;
- sauvegarder les données nettoyées dans `data/processed/`.

Les données nettoyées sont maintenant prêtes pour l'étape suivante :

**Notebook 03 — Data Integration**

L'objectif du prochain notebook sera de réunir les différentes tables afin de construire des datasets analytiques permettant d'étudier les patients, les exercices, les sessions, les scores et la progression.